In [2]:
import sys
!{sys.executable} -m pip install mysql-connector-python

In [3]:
import mysql.connector

conn = mysql.connector.connect(
    host="localhost",
    user="root",
    password="Kookoya@123?", 
    database="churn_project",
    port=3306
)

print("connected")

connected


In [4]:
import pandas as pd
#df=pd.read_excel('../data/Online Retail.xlsx')
#df.to_csv("../data/Online Retail.csv", index=False)
df = pd.read_csv('../data/Online Retail.csv')
df.head()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom


In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 541909 entries, 0 to 541908
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype  
---  ------       --------------   -----  
 0   InvoiceNo    541909 non-null  object 
 1   StockCode    541909 non-null  object 
 2   Description  540455 non-null  object 
 3   Quantity     541909 non-null  int64  
 4   InvoiceDate  541909 non-null  object 
 5   UnitPrice    541909 non-null  float64
 6   CustomerID   406829 non-null  float64
 7   Country      541909 non-null  object 
dtypes: float64(2), int64(1), object(5)
memory usage: 33.1+ MB


In [6]:
df=df.dropna(subset=['CustomerID'])

In [7]:
df=df[df['Quantity']>0]

In [8]:
df['InvoiceDate']=pd.to_datetime(df['InvoiceDate'])

In [9]:
df['TotalAmount'] = df['Quantity'] * df['UnitPrice']

In [10]:
snapshot_date = df['InvoiceDate'].max() + pd.Timedelta(days=1)

In [11]:
rfm = pd.read_sql("SELECT * FROM rfm", conn)

C:\Users\yasha\AppData\Local\Temp\ipykernel_12424\2678095461.py:1: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  rfm = pd.read_sql("SELECT * FROM rfm", conn)


In [12]:
rfm.columns = ['CustomerID', 'Recency' , 'Frequency', 'Monetary']

In [13]:
rfm.head()

,CustomerID,Recency,Frequency,Monetary
0,13047.0,31,10,3237.54
1,12583.0,2,15,7281.38
2,13748.0,95,5,948.25
3,15100.0,333,3,876.00
4,15291.0,25,15,4668.30


In [14]:
rfm.describe()

,Recency,Frequency,Monetary
count,4339.000000,4339.000000,4.339000e+03
mean,92.038258,4.600138,2.458328e+03
std,100.010502,22.943499,2.809265e+04
min,0.000000,1.000000,3.750000e+00
25%,17.000000,1.000000,3.074300e+02
50%,50.000000,2.000000,6.745200e+02
75%,141.500000,5.000000,1.662060e+03
max,373.000000,1428.000000,1.755277e+06


In [15]:
rfm['R_Score'] = pd.qcut(rfm['Recency'],4, labels=[4,3,2,1])
rfm['F_Score'] = pd.qcut(rfm['Frequency'].rank(method='first'),4, labels=[1,2,3,4])
rfm['M_Score'] = pd.qcut(rfm['Monetary'], 4 , labels=[1,2,3,4])

In [16]:
rfm['RFM_Score'] = rfm['R_Score'].astype(str) + rfm['F_Score'].astype(str) + rfm['M_Score'].astype(str)
rfm.head()

,CustomerID,Recency,Frequency,Monetary,R_Score,F_Score,M_Score,RFM_Score
0,13047.0,31,10,3237.54,3,4,4,344
1,12583.0,2,15,7281.38,4,4,4,444
2,13748.0,95,5,948.25,2,3,3,233
3,15100.0,333,3,876.00,1,3,3,133
4,15291.0,25,15,4668.30,3,4,4,344


In [17]:
def segment_customer(row):
    if row['R_Score'] == 4 and row['F_Score'] == 4:
        return 'Champions'
    elif row['R_Score'] >= 3 and row['F_Score'] >= 3:
        return 'Loyal Customers'
    elif row['R_Score'] <= 2 and row['F_Score'] >= 3:
        return 'At Risk'
    elif row['R_Score'] == 1:
        return 'Lost'
    else:
        return 'Average'
    
rfm['Segment'] = rfm.apply(segment_customer,axis=1)    

In [18]:
rfm['Segment'].value_counts()

Segment
Average            1249
Loyal Customers     948
Lost                921
At Risk             615
Champions           606
Name: count, dtype: int64

In [19]:
rfm['Segment'].value_counts(normalize=True) * 100

Segment
Average            28.785434
Loyal Customers    21.848352
Lost               21.226089
At Risk            14.173773
Champions          13.966352
Name: proportion, dtype: float64

In [20]:
rfm['Churn'] = (rfm['Recency'] > 30).astype(int)

In [21]:
rfm['Churn'].mean()

np.float64(0.6155796266420834)

churn: customers who have not made a purchase in the last 30 days are classified as churned. This allows indentification of inactive users and potential revenue loss.

In [22]:
rfm.groupby('Segment')['Churn'].mean()

Segment
At Risk            1.000000
Average            0.686149
Champions          0.000000
Lost               1.000000
Loyal Customers    0.293249
Name: Churn, dtype: float64

Insight: Churn by Customer Segment

The overall churn rate is approximately 62%, indicating a significant retention challenge.

Customers in the "Lost" and "At Risk" segments show extremely high churn rates (~100%), confirming they are no longer actively engaged.

"Average" customers also exhibit a high churn rate (~65%), suggesting a large portion of the customer base is vulnerable.

In contrast, "Champions" show no churn, highlighting their strong engagement and importance to the business.

### Churn Reduction Strategy

Based on the RFM segmentation and churn analysis, different customer groups require targeted strategies:

### 1. Champions (High Value, Highly Active)
These customers show 0% churn and contribute significantly to revenue.

**Strategy:**
- Introduce loyalty programs and exclusive offers  
- Provide early access to new products  
- Maintain high engagement to retain them  


### 2. Loyal Customers (Moderate Risk)
These customers are engaged but show some churn (~32%).

**Strategy:**
- Send personalized recommendations  
- Offer occasional discounts  
- Increase engagement through email campaigns  


### 3. Average Customers (High Risk Group)
This segment shows a high churn rate (~65%) and represents a large portion of users.

**Strategy:**
- Provide targeted discounts  
- Improve engagement through reminders  
- Highlight product value to retain them  


### 4. At-Risk Customers (Critical)
These customers are highly likely to churn (~100%).

**Strategy:**
- Send urgent re-engagement campaigns  
- Offer strong incentives (discounts, bundles)  
- Identify reasons for disengagement  


### 5. Lost Customers
These users have already churned.

**Strategy:**
- Launch win-back campaigns  
- Offer special comeback discounts  
- Analyze why they left to prevent future churn

In [23]:
df['Month'] = df['InvoiceDate'].dt.to_period('M')
monthly_users = df.groupby('Month')['CustomerID'].nunique()
monthly_users.head()

Month
2010-12    885
2011-01    741
2011-02    758
2011-03    974
2011-04    856
Freq: M, Name: CustomerID, dtype: int64

## Insight: Monthly Customer Trend

The number of active customers shows fluctuations over time. A noticeable drop is observed from December to January, likely due to seasonal effects or poor retention after peak activity.

Although customer count recovers in subsequent months, the inconsistency suggests weak customer retention, with users not consistently returning over time.

In [24]:
df['FirstPurchaseMonth'] = df.groupby('CustomerID')['Month'].transform('min')
df['MonthDiff'] = (df['Month'] - df['FirstPurchaseMonth']).apply(lambda x: x.n)

In [25]:
cohort = df.groupby(['FirstPurchaseMonth', 'MonthDiff'])['CustomerID'].nunique().reset_index()
cohort.head()

,FirstPurchaseMonth,MonthDiff,CustomerID
0,2010-12,0,885
1,2010-12,1,324
2,2010-12,2,286
3,2010-12,3,340
4,2010-12,4,321


In [26]:
cohort_pivot= cohort.pivot(index='FirstPurchaseMonth',columns='MonthDiff', values='CustomerID')
cohort_pivot.head()

MonthDiff,0,1,2,3,4,5,6,7,8,9,10,11,12
FirstPurchaseMonth,,,,,,,,,,,,,
2010-12,885.0,324.0,286.0,340.0,321.0,352.0,321.0,309.0,313.0,350.0,331.0,445.0,235.0
2011-01,417.0,92.0,111.0,96.0,134.0,120.0,103.0,101.0,125.0,136.0,152.0,49.0,NaN
2011-02,380.0,71.0,71.0,108.0,103.0,94.0,96.0,106.0,94.0,116.0,26.0,NaN,NaN
2011-03,452.0,68.0,114.0,90.0,101.0,76.0,121.0,104.0,126.0,39.0,NaN,NaN,NaN
2011-04,300.0,64.0,61.0,63.0,59.0,68.0,65.0,78.0,22.0,NaN,NaN,NaN,NaN


In [27]:
cohort_size = cohort_pivot[0]
retention = cohort_pivot.divide(cohort_size,axis=0)
retention.head()

MonthDiff,0,1,2,3,4,5,6,7,8,9,10,11,12
FirstPurchaseMonth,,,,,,,,,,,,,
2010-12,1.0,0.366102,0.323164,0.384181,0.362712,0.397740,0.362712,0.349153,0.353672,0.395480,0.374011,0.502825,0.265537
2011-01,1.0,0.220624,0.266187,0.230216,0.321343,0.287770,0.247002,0.242206,0.299760,0.326139,0.364508,0.117506,NaN
2011-02,1.0,0.186842,0.186842,0.284211,0.271053,0.247368,0.252632,0.278947,0.247368,0.305263,0.068421,NaN,NaN
2011-03,1.0,0.150442,0.252212,0.199115,0.223451,0.168142,0.267699,0.230088,0.278761,0.086283,NaN,NaN,NaN
2011-04,1.0,0.213333,0.203333,0.210000,0.196667,0.226667,0.216667,0.260000,0.073333,NaN,NaN,NaN,NaN


## Root Cause of Churn

Retention analysis shows that a large number of customers do not return after their first purchase. Only around 30–40% of users come back in the following month.

This indicates that the primary driver of churn is poor early-stage engagement, where users fail to see enough value to continue using the service.

| Segment   | Problem             | Action             |
| --------- | ------------------- | ------------------ |
| Champions | High value          | loyalty rewards    |
| Loyal     | maintain engagement | upsell             |
| At Risk   | inactivity          | targeted offers    |
| Lost      | churned             | win-back campaigns |


In [28]:
rfm.to_csv("rfm_final.csv", index=False)

In [29]:
retention.to_csv("retention.csv")